# Gemma 3 Text-to-SQL: LoRA Fine-Tuning with Open-RL

This notebook is a small guide to two ideas:
1. how Tinker talks to a running Open-RL backend for both sampling and training
2. how a short LoRA loop changes model behavior on a text-to-SQL task

The real thing to pay attention to is the training loop and the backend interaction model. We will use the same backend three ways: sample the untouched model, train a LoRA adapter, then sample again from the updated adapter.

The code stays intentionally explicit. Instead of hiding the interesting parts behind a helper library, the notebook shows the prompt format, the backend calls, the adapter training loop, and the before/after sampling path in plain code.

Notebook flow:
1. load the dataset and inspect a few concrete rows
2. connect Tinker to the backend and build prompt/completion examples
3. sample the untouched model through the backend
4. run a short LoRA training loop
5. sample again and keep tinkering through the same backend

| | |
|---|---|
| **Model** | `google/gemma-3-1b-pt` |
| **Dataset** | [gretel-synthetic-text-to-sql](https://huggingface.co/datasets/philschmid/gretel-synthetic-text-to-sql) (100k examples) |
| **Method** | LoRA SFT, attention + MLP adapters, base frozen |
| **Infra** | [Open-RL](https://github.com/gke-labs/open-rl) server + [Tinker SDK](https://pypi.org/project/tinker/) |


## Prerequisites

Before running this notebook, you need:

**1. Client dependencies**
```bash
cd client && uv sync
```

**2. A running Open-RL server** in a separate terminal

| Hardware | Command |
|---|---|
| Local single-process run | `make server BASE_MODEL=google/gemma-3-1b-pt` |
| GPU + vLLM | `make vllm BASE_MODEL=google/gemma-3-1b-pt` in terminal 1 and `make server BASE_MODEL=google/gemma-3-1b-pt SAMPLER=vllm` in terminal 2 |

**3. Hugging Face access** to `google/gemma-3-1b-pt`. Run `uv run hf auth login` if needed.

The same backend will be used three ways in this notebook: first for baseline sampling, then for LoRA training, and finally for interactive post-training sampling.


In [ ]:
import json
import os
import random
import re
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import tinker
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from tinker import types

os.environ.setdefault("TINKER_API_KEY", "tml-dummy-key")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## Config

These defaults are tuned for a short local run on this Mac. If you are just playing with the notebook, the first knobs worth changing are `steps`, `eval_every`, the example ID lists below, and the playground inputs.


In [ ]:

DEFAULT_BASE_URL = os.getenv("TINKER_BASE_URL") or os.getenv("OPEN_RL_BASE_URL") or "http://127.0.0.1:9003"
DATASET_NAME = "philschmid/gretel-synthetic-text-to-sql"
MAX_SEQ_LENGTH = 512
EVAL_MAX_TOKENS = 256
RAW_DATASET_LIMIT = 12_500
EVAL_SPLIT_SIZE = 2_500


@dataclass
class Config:
    base_model: str = "google/gemma-3-1b-pt"
    tokenizer_model: str = "google/gemma-3-1b-it"
    steps: int = 25
    batch_size: int = 8
    lora_rank: int = 16
    learning_rate: float = 2e-4
    server_command: str = "make server BASE_MODEL=google/gemma-3-1b-pt"
    base_url: str = DEFAULT_BASE_URL
    grad_clip_norm: float = 0.3
    eval_every: int = 12
    train_limit: int = 10_000
    eval_limit: int = 5
    seed: int = 30
    metrics_path: str = "artifacts/texttosql_notebook_metrics.jsonl"
    plot_path: str = "artifacts/texttosql_notebook_results.png"
    run_label: str = "Gemma 3 Text-to-SQL SFT"


config = Config()
metrics_path = Path(config.metrics_path)
plot_path = Path(config.plot_path)

config_table = pd.DataFrame.from_dict(asdict(config), orient="index", columns=["value"])
display(config_table)


## 1. Load the Dataset and Pick a Few Concrete Rows

If you are new to this dataset, do not skip this section. Spend a minute looking at a few rows until the pattern is obvious: each example pairs a natural-language question, a schema block, and one target SQL answer.

The goal here is not exhaustive exploration. It is just to build intuition before we ask the backend to sample or train.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_model)

dataset = load_dataset(DATASET_NAME, split="train")
dataset = dataset.shuffle(seed=config.seed)
dataset = dataset.select(range(min(RAW_DATASET_LIMIT, len(dataset))))
split = dataset.train_test_split(test_size=min(EVAL_SPLIT_SIZE, len(dataset) - 1), shuffle=False)

print(f"Dataset loaded: {len(dataset):,} rows")
print(f"Training split: {len(split['train']):,} rows")
print(f"Eval split: {len(split['test']):,} rows")

In [ ]:
DATASET_PREVIEW_IDS = [0, 1, 2, 3]
BASELINE_EXAMPLE_IDS = [0, 1]
COMPARISON_EXAMPLE_IDS = [2, 3, 4]
PLAYGROUND_PREVIEW_COUNT = 1

PLAYGROUND_SCHEMA = """
CREATE TABLE employees (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    department VARCHAR(50),
    salary DECIMAL(10,2),
    hire_date DATE
);
""".strip()

PLAYGROUND_QUESTIONS = [
    "What are the names and salaries of employees in the Engineering department, ordered by salary descending?",
    "How many employees were hired after 2023-01-01?",
    "What is the average salary by department?",
]

dataset_preview_rows = [split["train"][example_id] for example_id in DATASET_PREVIEW_IDS]
dataset_preview_df = pd.DataFrame(
    [
        {
            "question": row["sql_prompt"],
            "schema": row["sql_context"],
            "target_sql": row["sql"],
        }
        for row in dataset_preview_rows
    ]
)
print("A few raw training rows")
display(dataset_preview_df)

selected_eval_rows = []
for label, example_ids in [
    ("Base model feel", BASELINE_EXAMPLE_IDS),
    ("Before vs after compare", COMPARISON_EXAMPLE_IDS),
]:
    for example_id in example_ids:
        row = split["test"][example_id]
        selected_eval_rows.append(
            {
                "group": label,
                "eval_example": example_id,
                "question": row["sql_prompt"],
                "target_sql": row["sql"],
            }
        )

print("The exact eval questions we will revisit later")
display(pd.DataFrame(selected_eval_rows))

print("Edit DATASET_PREVIEW_IDS, BASELINE_EXAMPLE_IDS, COMPARISON_EXAMPLE_IDS, PLAYGROUND_SCHEMA, PLAYGROUND_QUESTIONS, or PLAYGROUND_PREVIEW_COUNT here, then rerun the sections you care about.")


## 2. Connect Tinker to the Open-RL Backend

This notebook is really about one workflow: use Tinker to talk to a running Open-RL backend, sample from the current model state, run a few training steps, and sample again.

Mental model:
- `tinker.ServiceClient(...)` is the notebook's main handle to the backend.
- `create_lora_training_client_async(...)` creates a trainable adapter session on top of the base model already loaded by the server.
- `save_weights_for_sampler(...)` lets us turn the current adapter state into a sampling client, so the same backend can power evaluation and free-form playground prompts.


In [ ]:
async def get_server_model(service_client):
    try:
        capabilities = await service_client.get_server_capabilities_async()
    except Exception as exc:
        message = (
            f"Open-RL server at {config.base_url} is not reachable. "
            f"Start it with `{config.server_command}`."
        )
        raise RuntimeError(message) from exc

    for model in capabilities.supported_models:
        model_name = getattr(model, "model_name", None)
        if model_name:
            return model_name

    return None


service_client = tinker.ServiceClient(
    api_key=os.getenv("TINKER_API_KEY", "tml-dummy-key"),
    base_url=config.base_url,
)

server_model = await get_server_model(service_client)
if server_model is None:
    raise RuntimeError(
        f"Server at {config.base_url} is reachable, but no model is loaded. "
        f"Start it with `{config.server_command}`."
    )

print(f"Connected to {config.base_url}")
print(f"Server model: {server_model}")


## 3. Turn Dataset Rows into Prompt/Completion Training Examples

Each dataset row becomes one chat-style prompt plus one target SQL completion. The exact same prompt template is used for training, evaluation, and the interactive playground later.

The only loss-weighting detail that matters is simple: prompt tokens get weight `0` and assistant completion tokens get weight `1`. That keeps the training loop focused on the SQL generation target.


In [ ]:
PROMPT_TEMPLATE = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{context}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>
"""


def build_messages(sample):
    return [
        {
            "role": "user",
            "content": PROMPT_TEMPLATE.format(
                question=sample["sql_prompt"],
                context=sample["sql_context"],
            ),
        },
        {
            "role": "assistant",
            "content": sample["sql"],
        },
    ]


def make_datum(token_ids, weights):
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=token_ids[:-1]),
        loss_fn_inputs={
            "weights": weights[1:],
            "target_tokens": token_ids[1:],
        },
    )


def build_example(sample):
    messages = build_messages(sample)

    prompt_text = tokenizer.apply_chat_template(
        messages[:1],
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    full_token_ids = tokenizer.encode(full_text, add_special_tokens=False)

    if len(full_token_ids) <= len(prompt_token_ids):
        return None
    if len(full_token_ids) > MAX_SEQ_LENGTH:
        return None

    target_token_count = len(full_token_ids) - len(prompt_token_ids)
    # Only train on the assistant completion, not the prompt prefix.
    weights = [0] * len(prompt_token_ids) + [1] * target_token_count

    return {
        "question": sample["sql_prompt"],
        "schema": sample["sql_context"],
        "target": sample["sql"],
        "prompt_tokens": prompt_token_ids,
        "active_tokens": target_token_count,
        "datum": make_datum(full_token_ids, weights),
    }


def build_examples(dataset_split, limit):
    examples = []
    limited_split = dataset_split.select(range(min(limit, len(dataset_split))))

    for row in limited_split:
        example = build_example(row)
        if example is not None:
            examples.append(example)

    return examples


train_examples = build_examples(split["train"], config.train_limit)
eval_examples = build_examples(split["test"], config.eval_limit)

if not train_examples:
    raise RuntimeError("No training examples fit within MAX_SEQ_LENGTH.")

batch_size = min(config.batch_size, len(train_examples))
batches_per_epoch = (len(train_examples) + batch_size - 1) // batch_size

print(f"Usable training examples: {len(train_examples):,}")
print(f"Usable eval examples: {len(eval_examples):,}")
print(f"Batch size: {batch_size}")
print(f"Approximate batches per epoch: {batches_per_epoch:,}")


## 4. Define Sampling and Evaluation Helpers

Every evaluation pass works by snapshotting the current adapter state into a sampler, sending the eval prompts to the backend, and comparing the generated SQL with the reference answers.

That is the same interaction pattern we will use again in the playground. The only thing that changes over time is the adapter state behind the sampler.


In [ ]:
def normalize_sql(text):
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    text = text.replace("<|im_start|>", " ")
    text = text.replace("<|im_end|>", " ")
    text = text.strip()

    prefixes_to_strip = [
        r"^assistant\s*[:\-]?\s*",
        r"^sql\s*[:\-]?\s*",
        r"^```(?:sql)?\s*",
    ]
    for pattern in prefixes_to_strip:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    text = re.sub(r"\s*```$", "", text)
    text = " ".join(text.split()).lower()
    text = re.sub(r"\s+([,;()])", r"\1", text)
    text = re.sub(r"([,(])\s+", r"\1", text)
    return text


def create_sampler(alias):
    # Snapshot the current adapter state and reopen it through the sampling API.
    weights_path = training_client.save_weights_for_sampler(name=alias).result().path
    return service_client.create_sampling_client(weights_path)


def decode_first_sequence(result):
    if result.sequences:
        token_ids = result.sequences[0].tokens
    else:
        token_ids = []
    return tokenizer.decode(token_ids, skip_special_tokens=True)


def pick_examples(examples, example_ids):
    selected_examples = []
    for example_id in example_ids:
        if example_id < 0 or example_id >= len(examples):
            raise IndexError(f"Example id {example_id} is out of range for {len(examples)} eval examples")
        selected_examples.append(examples[example_id])
    return selected_examples


def run_eval(alias, examples):
    sampler = create_sampler(alias)
    pending_requests = []

    for example in examples:
        future = sampler.sample(
            prompt=types.ModelInput.from_ints(tokens=example["prompt_tokens"]),
            num_samples=1,
            sampling_params=types.SamplingParams(
                max_tokens=EVAL_MAX_TOKENS,
                temperature=0.0,
            ),
        )
        pending_requests.append((example, future))

    rows = []
    exact_match_count = 0
    similarity_sum = 0.0

    for example, future in pending_requests:
        result = future.result()
        raw_prediction = decode_first_sequence(result).strip()
        predicted_sql = normalize_sql(raw_prediction)
        target_sql = normalize_sql(example["target"])
        similarity = SequenceMatcher(None, predicted_sql, target_sql).ratio()
        exact_match = predicted_sql == target_sql

        if exact_match:
            exact_match_count += 1
        similarity_sum += similarity

        rows.append(
            {
                "question": example["question"],
                "schema": example["schema"],
                "target_raw": example["target"],
                "predicted_raw": raw_prediction,
                "predicted": predicted_sql,
                "target": target_sql,
                "exact": exact_match,
                "similarity": similarity,
            }
        )

    details_df = pd.DataFrame(rows)
    row_count = max(1, len(rows))
    exact_match_rate = exact_match_count / row_count
    average_similarity = similarity_sum / row_count

    return exact_match_rate, average_similarity, details_df


def show_eval_examples(title, details_df):
    print(title)
    display(
        details_df[["question", "target_raw", "predicted_raw", "exact", "similarity"]].rename(
            columns={
                "target_raw": "target_sql",
                "predicted_raw": "model_output",
            }
        )
    )


def show_eval_comparison(title, before_df, after_df):
    if len(before_df) != len(after_df):
        raise ValueError("Before and after comparison tables must have the same number of rows")

    comparison_df = pd.DataFrame(
        {
            "question": before_df["question"],
            "target_sql": before_df["target_raw"],
            "base_output": before_df["predicted_raw"],
            "trained_output": after_df["predicted_raw"],
            "base_exact": before_df["exact"],
            "trained_exact": after_df["exact"],
            "base_similarity": before_df["similarity"],
            "trained_similarity": after_df["similarity"],
        }
    )
    print(title)
    display(comparison_df)


## 5. Sample the Untuned Model Through the Backend

This is the first moment where the backend story becomes concrete. We create a LoRA training client, but we have not updated any weights yet, so the first sampler is still giving us the untouched baseline model at step `0`.

Looking at a couple real examples here matters more than staring at one scalar metric. You want a feel for what the base model gets roughly right, where it is brittle, and what kinds of SQL errors show up before training.


In [ ]:
training_client = await service_client.create_lora_training_client_async(
    base_model=config.base_model,
    rank=config.lora_rank,
    train_mlp=True,
    train_attn=True,
    train_unembed=False,
)

print(f"Created LoRA training client | base_model={config.base_model} | rank={config.lora_rank}")

before_exact, before_similarity, baseline_df = run_eval("baseline", eval_examples)

baseline_summary_df = pd.DataFrame(
    [
        {"metric": "Exact match", "value": f"{before_exact * 100:.1f}%"},
        {"metric": "Similarity", "value": f"{before_similarity * 100:.1f}%"},
        {"metric": "Eval rows", "value": len(eval_examples)},
    ]
)
display(baseline_summary_df)

_, _, baseline_examples_df = run_eval("baseline_examples", pick_examples(eval_examples, BASELINE_EXAMPLE_IDS))
show_eval_examples("A couple of concrete untuned examples", baseline_examples_df)

_, _, comparison_before_df = run_eval("comparison_before", pick_examples(eval_examples, COMPARISON_EXAMPLE_IDS))


## 6. Run a Short LoRA Training Loop

This is the core of the notebook. Each step does four things:
1. pick a small batch of prompt/completion examples
2. send their `Datum` objects to the backend with `forward_backward_async(...)`
3. apply one Adam update with `optim_step_async(...)`
4. occasionally snapshot the current adapter into a sampler and evaluate it

Nothing fancy is hidden here. The point is to show the exact loop you would reason about if you were debugging training, changing the loss cadence, or swapping in a different batch construction strategy.

With the default Mac-friendly settings, we record metrics at step `0`, step `12`, and step `25`. That gives us a beginning, middle, and end view without making the notebook noisy.


In [ ]:
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text("", encoding="utf-8")

metrics_log = []


def record_metric(step, phase, loss=None, exact_match=None, similarity=None):
    row = {"step": step, "phase": phase}
    if loss is not None:
        row["loss"] = loss
    if exact_match is not None:
        row["exact_match"] = exact_match
    if similarity is not None:
        row["similarity"] = similarity

    metrics_log.append(row)
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")


record_metric(step=0, phase="eval", exact_match=before_exact, similarity=before_similarity)
print(f"Metrics will be written to {metrics_path}")

losses = []
eval_history = [
    {
        "step": 0,
        "exact_match": before_exact,
        "similarity": before_similarity,
    }
]

rng = random.Random(config.seed)
shuffled_indices = list(range(len(train_examples)))
rng.shuffle(shuffled_indices)
position = 0

progress_bar = tqdm(range(1, config.steps + 1), desc="Training", unit="step")

for step in progress_bar:
    if position + batch_size > len(shuffled_indices):
        rng.shuffle(shuffled_indices)
        position = 0

    batch_indices = shuffled_indices[position:position + batch_size]
    position += batch_size

    batch = []
    for index in batch_indices:
        batch.append(train_examples[index])

    datums = []
    active_token_count = 0
    for example in batch:
        datums.append(example["datum"])
        active_token_count += example["active_tokens"]

    # The backend does the backward pass against the current LoRA adapter.
    forward_backward_future = await training_client.forward_backward_async(datums, "cross_entropy")
    optimizer_future = await training_client.optim_step_async(
        types.AdamParams(
            learning_rate=config.learning_rate,
            grad_clip_norm=config.grad_clip_norm,
        )
    )

    forward_backward_result = await forward_backward_future
    await optimizer_future

    loss_sum = float(forward_backward_result.metrics.get("loss:sum", 0.0))
    loss = loss_sum / max(1, active_token_count)
    losses.append(loss)
    record_metric(step=step, phase="train", loss=loss)
    progress_bar.set_postfix(loss=f"{loss:.4f}")

    # Step 0 was recorded before the loop; these evals capture the middle and end.
    should_evaluate = step % config.eval_every == 0 or step == config.steps
    if should_evaluate:
        exact_match, similarity, _ = run_eval(f"step_{step}", eval_examples)
        eval_history.append(
            {
                "step": step,
                "exact_match": exact_match,
                "similarity": similarity,
            }
        )
        record_metric(
            step=step,
            phase="eval",
            exact_match=exact_match,
            similarity=similarity,
        )
        progress_bar.write(
            f"[eval at step {step}] exact={exact_match * 100:.1f}% similarity={similarity * 100:.1f}%"
        )


## 7. Summarize the Run

Keep the output compact: one table for the before/after metrics and one chart for the training trace. That keeps the notebook readable in GitHub while still showing whether the loop is moving in the right direction.


In [ ]:
eval_df = pd.DataFrame(eval_history)
eval_plot_df = eval_df.copy()
eval_plot_df["exact_match"] = eval_plot_df["exact_match"] * 100
eval_plot_df["similarity"] = eval_plot_df["similarity"] * 100

initial_loss = losses[0]
final_loss = losses[-1]
loss_drop_percent = (initial_loss - final_loss) / (abs(initial_loss) or 1.0) * 100

summary_rows = [
    {
        "Metric": "Exact Match (%)",
        "Before": f"{before_exact * 100:.1f}",
        "After": f"{eval_history[-1]['exact_match'] * 100:.1f}",
    },
    {
        "Metric": "Similarity (%)",
        "Before": f"{before_similarity * 100:.1f}",
        "After": f"{eval_history[-1]['similarity'] * 100:.1f}",
    },
    {
        "Metric": "Loss",
        "Before": f"{initial_loss:.4f}",
        "After": f"{final_loss:.4f}",
    },
    {
        "Metric": "Loss Drop (%)",
        "Before": "-",
        "After": f"{loss_drop_percent:.1f}",
    },
]

summary_df = pd.DataFrame(summary_rows).set_index("Metric")
display(summary_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

loss_steps = list(range(1, len(losses) + 1))
axes[0].plot(loss_steps, losses, linewidth=0.8, alpha=0.8)
if len(losses) > 1:
    smoothing_window = min(25, max(1, len(losses) // 4))
    smoothed_loss = pd.Series(losses).rolling(smoothing_window, min_periods=1).mean()
    axes[0].plot(
        loss_steps,
        smoothed_loss,
        color="red",
        linewidth=1.5,
        label=f"{smoothing_window}-step average",
    )
    axes[0].legend()
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].grid(alpha=0.2)

axes[1].plot(eval_plot_df["step"], eval_plot_df["exact_match"], "o-", color="#2ca02c")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Percent")
axes[1].set_title("Exact Match")
axes[1].grid(alpha=0.2)

axes[2].plot(eval_plot_df["step"], eval_plot_df["similarity"], "o-", color="#ff7f0e")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Percent")
axes[2].set_title("Sequence Similarity")
axes[2].grid(alpha=0.2)

fig.suptitle(config.run_label, fontsize=13)
plt.tight_layout()

plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved plot to {plot_path}")


## 8. Compare Fresh Examples Before and After Training

Fresh examples are the easiest sanity check. If the model looks better only on the first examples you stared at before training, that does not tell you much. Looking at a different slice makes the improvement feel more real.

This section reuses the same backend sampling path as before, just with the updated adapter state.


In [ ]:
after_exact, after_similarity, after_training_df = run_eval("final", eval_examples)

after_summary_df = pd.DataFrame(
    [
        {"metric": "Exact match", "before": f"{before_exact * 100:.1f}%", "after": f"{after_exact * 100:.1f}%"},
        {"metric": "Similarity", "before": f"{before_similarity * 100:.1f}%", "after": f"{after_similarity * 100:.1f}%"},
    ]
)
display(after_summary_df)

_, _, comparison_after_df = run_eval("comparison_after", pick_examples(eval_examples, COMPARISON_EXAMPLE_IDS))
show_eval_comparison("Before vs after on a fresh set of eval examples", comparison_before_df, comparison_after_df)

## 9. Keep Tinkering Through the Backend

This final section turns the notebook into a small client for your backend. Edit `PLAYGROUND_SCHEMA` or `PLAYGROUND_QUESTIONS`, rerun this section, and Tinker will sample from the current trained adapter.

This is the part that should feel playable: you can try your own schema, ask a slightly different question, or train a bit longer and immediately see how the adapter responds.


In [ ]:
def build_user_prompt(question, schema):
    return PROMPT_TEMPLATE.format(question=question, context=schema)


def sample_sql(question, schema, sampler_name="interactive"):
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(question, schema)}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompt_token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    sampler = create_sampler(sampler_name)

    result = sampler.sample(
        prompt=types.ModelInput.from_ints(tokens=prompt_token_ids),
        num_samples=1,
        sampling_params=types.SamplingParams(
            max_tokens=EVAL_MAX_TOKENS,
            temperature=0.0,
        ),
    ).result()

    return decode_first_sequence(result).strip()


def make_generated_sql_row(question, schema, sampler_name="interactive"):
    generated_sql = sample_sql(question, schema, sampler_name=sampler_name)
    return {
        "question": question,
        "generated_sql": generated_sql or "-- empty --",
    }


print("Schema")
print(PLAYGROUND_SCHEMA)

generated_rows = [
    make_generated_sql_row(question, PLAYGROUND_SCHEMA, sampler_name="interactive")
    for question in PLAYGROUND_QUESTIONS[:PLAYGROUND_PREVIEW_COUNT]
]
display(pd.DataFrame(generated_rows))


## Next Steps

- Increase `config.steps` once the short Mac run looks sane.
- Sweep `lora_rank`, `learning_rate`, and `batch_size` to see how the adapter changes.
- Swap in your own schema and questions in the playground once the before/after examples look reasonable.
- Keep `config.base_model = "google/gemma-3-1b-pt"` and start `make server BASE_MODEL=google/gemma-3-1b-pt` for the default Gemma path.
- Use `make server BASE_MODEL=google/gemma-3-1b-pt-gpu` if you want a faster backend.
- Run `cd client && uv run python texttosql_sft.py gemma` for the non-notebook script path.
- Plot saved metrics with `make plot-metrics FILE=client/artifacts/texttosql_notebook_metrics.jsonl`.
